Suppose we want to do symmetry expansion on some VQE circuit or other state generated by an arbitrary circuit and we want to add additional stabilizers to detect additional errors. The gates in that circuit are not logical operations of the code defined by the symmetry group, so there is no clear way to compile this circuit into a circuit for the new code. However, suppose there exists a circuit that maps codewords of the original code to the equivalent codewords of the new code. Then we can apply the encoding circuit after the computation to get a codeword of the new code.

We will carry out the following steps with a repetition code:
1. Encode the logical logical $|0\rangle$ state of the original repetition code (stabilized by $ZZI$ and $IZZ$).
2. Carry out a logical gate on the encoded qubit.
3. Apply the circuit that converts codewords of the original repetition code to that of the new code.
4. Measure a logical observable of the new code, mitigating errors using symmetry expansion under the stabilizer group of the new code.

In this case we will prepare the $|\bar{+}\rangle$ state then apply a logical phase gate to get the $Y$-eigenstate $|+i\rangle$.

In [13]:
from typing import List
import itertools
import functools
import numpy as np
import matplotlib.pyplot as plt
import cirq
from encoded.repetition_code import (
    generate_stabilizers,
    encoding_repetition
)
from encoded.expanded_repetition_code import (
    x_bar,
    z_bar,
    stabilizer_generators,
    encoding_expanded_repetition,
    U_XXX
)
from encoded.symmetry_expansion import symmetry_expansion

In [14]:
circuit = cirq.Circuit()
encoding_ckt, qs = encoding_repetition('+', 3)

logical_s_circuit = cirq.Circuit()
logical_s_circuit.append(cirq.S(qs[0]))

circuit += encoding_ckt
circuit += logical_s_circuit
circuit += U_XXX
print(circuit)

0: ───H───@───────S───H───@───H───────────
          │               │
1: ───────X───@───────H───┼───@───H───────
              │           │   │
2: ───────────X───────H───┼───┼───@───H───
                          │   │   │
3: ───────────────────────X───X───X───H───


In [12]:
# Define logical operators of the original repetition code.
logical_x = cirq.X.on(qs[0]) * cirq.X.on(qs[1]) * cirq.X.on(qs[2])
logical_z = cirq.Z.on(qs[0])
logical_y = 1.0j * logical_x * logical_z
# Define Y for the new code.
y_bar = 1.0j * x_bar * z_bar

In [15]:
sim = cirq.Simulator()
result = sim.simulate_expectation_values(circuit, y_bar)
print(result[0])

(1+0j)
